# RWF-2000 dataset preparation

Run the single cell below, then **stop**. It is self-contained:
no internet, no pip, no py7zr. It verifies all 13 archive volumes,
extracts, recovers the 24 over-long CJK filenames under safe names,
resolves the nested `RWF-2000/RWF-2000` path, reconciles disk against
the archive table, and prints one verdict.

Do **not** start R3D-18 training or download Kinetics-400 weights here.


In [ ]:
# =====================================================================
# RWF-2000 KAGGLE BOOTSTRAP -- run this ONE cell, then stop.
#
# Self-contained: the project modules it needs are embedded below as a
# compressed payload. It needs NO internet, NO PyPI packages (py7zr in
# particular is never used), and no access to the local source tree.
# Only `cv2` and `numpy` are needed, both preinstalled on Kaggle.
#
# Generated by tools/build_kaggle_cell.py -- do not edit by hand.
# =====================================================================
import base64, io, os, shutil, subprocess, sys, zipfile, zlib
from pathlib import Path

# Kaggle defaults; every one can be overridden with an environment
# variable so the same cell can be exercised outside Kaggle.
SRC_DIR       = Path(os.environ.get("RWF_SRC_DIR", "/kaggle/working/_rwf2000_src"))
TARGET        = Path(os.environ.get("RWF_TARGET", "/kaggle/working/rwf2000"))
METADATA      = Path(os.environ.get("RWF_METADATA", "/kaggle/working/rwf2000_metadata"))
SEARCH_ROOTS  = os.environ.get("RWF_SEARCH_ROOTS", "/kaggle/input").split(os.pathsep)
STALE_OUTPUTS = os.environ.get("RWF_STALE", "/kaggle/working/RWF-2000").split(os.pathsep)
CHECK_MD5     = os.environ.get("RWF_SKIP_MD5", "") == ""
FRESH         = os.environ.get("RWF_FRESH", "") != ""

PAYLOAD = "..."


def unpack_modules() -> Path:
    """Write the embedded project modules to disk and return their directory."""
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    blob = zlib.decompress(base64.b64decode(PAYLOAD))
    with zipfile.ZipFile(io.BytesIO(blob)) as archive:
        archive.extractall(SRC_DIR)
    return SRC_DIR


def ensure_seven_zip(find_7z) -> str:
    """Return a usable 7-Zip binary, installing p7zip only if necessary.

    Discovery itself is delegated to the project's own ``find_7z`` so the
    rule lives in one place rather than being restated here.
    """
    found = find_7z()
    if found:
        return found
    print("7z not present; attempting a system install (needs network) ...")
    for command in (
        ["apt-get", "-qq", "install", "-y", "p7zip-full"],
        ["apt-get", "-qq", "install", "-y", "p7zip"],
    ):
        try:
            subprocess.run(command, check=False, timeout=300)
        except Exception:
            pass
        found = find_7z()
        if found:
            return found
    raise SystemExit(
        "BLOCKER: no 7-Zip binary is available and p7zip could not be "
        "installed (no network). Attach a p7zip .deb as a Kaggle dataset, "
        "or enable internet for this notebook once."
    )


def drop_stale_outputs() -> None:
    """Remove a previous FAILED extraction that used a different path.

    Only paths named explicitly in STALE_OUTPUTS are touched, and never
    the current target -- a complete dataset is never thrown away here.
    """
    for raw in STALE_OUTPUTS:
        stale = Path(raw.strip())
        if not raw.strip() or not stale.is_dir():
            continue
        if stale.resolve() == TARGET.resolve():
            continue
        print(f"Removing previous incomplete extraction: {stale}")
        shutil.rmtree(stale, ignore_errors=True)


print("=" * 70)
print("RWF-2000 KAGGLE BOOTSTRAP")
print("=" * 70)

source = unpack_modules()
print(f"modules      : {source}")
sys.path.insert(0, str(source))

from rwf2000_kaggle import (  # noqa: E402
    KagglePreparationError,
    bootstrap,
    find_7z,
)

seven_zip = ensure_seven_zip(find_7z)
print(f"7z binary    : {seven_zip}")

drop_stale_outputs()

try:
    ready, report = bootstrap(
        target=TARGET,
        search_roots=SEARCH_ROOTS,
        metadata_dir=METADATA,
        check_md5=CHECK_MD5,
        fresh=FRESH,
        validate=True,
    )
except KagglePreparationError as error:
    print(f"\nBLOCKER: {error}")
    raise SystemExit(1)

print("\n" + "=" * 70)
print("RESULT: READY" if ready else "RESULT: NOT READY")
print("=" * 70)
if ready:
    print(f"dataset root : {report['dataset_root']}")
    print(f"metadata     : {report['manifest_path']}")
    print("\nSTOP HERE. Do not start training in this cell.")
else:
    print("The reconciliation/validation output above names the exact problem.")
